# Parkinson's Disease Detection using Biomedical Voice Measurements
> **End-to-End Machine Learning Pipeline with Subject-Aware Cross-Validation**

---

### Project Overview
Parkinson's disease (PD) is a progressive neurodegenerative disorder affecting motor control and vocal acoustics. This notebook develops a non-invasive machine learning classification system to distinguish individuals with Parkinson's disease from healthy controls using 22 biomedical voice measurement features.

### Key Methodology & Engineering Highlights:
- **Data Leakage Prevention (Subject-Aware Splitting)**: Each individual in the dataset provides multiple voice recordings (typically 6-7 replicates). Naive random train/test splitting leaks patient-specific acoustic traits across folds, yielding artificially inflated metrics. We extract unique patient identifiers and enforce strict **subject-level group splitting**.
- **Unified Pipeline Architecture (`StandardScaler` + `Linear SVC`)**: Support Vector Classifiers are sensitive to feature scales. Encapsulating preprocessing and classification into a Scikit-Learn `Pipeline` guarantees that feature scaling parameters are learned strictly from training folds.
- **Comprehensive Validation**: The model is evaluated on held-out unseen subjects and rigorously benchmarked using **5-Fold `StratifiedGroupKFold` Cross-Validation**.

---

## 1. Environment Setup & Dependencies
Import essential libraries for data manipulation, machine learning pipeline construction, cross-validation, and performance evaluation.

In [1]:
import numpy as np
import pandas as pd
import joblib

# Scikit-Learn pipeline & preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn import svm

# Model selection & evaluation metrics
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedGroupKFold
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Configuration for clean DataFrame rendering
pd.set_option('display.max_columns', None)


## 2. Data Ingestion & Exploratory Data Analysis (EDA)
Load the Parkinson's biomedical voice dataset and inspect its structure, dimensions, data types, missing values, and summary statistics.

In [2]:
# Load the dataset
parkinsons_data = pd.read_csv('../Dataset/parkinsons.csv')

print(f"Dataset Shape: {parkinsons_data.shape[0]} rows, {parkinsons_data.shape[1]} columns")
print(f"Total Missing Values: {parkinsons_data.isnull().sum().sum()}")
parkinsons_data.head()


Dataset Shape: 195 rows, 24 columns
Total Missing Values: 0


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,MDVP:APQ,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,0.426,0.02182,0.03130,0.02971,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,0.626,0.03134,0.04518,0.04368,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,0.482,0.02757,0.03858,0.03590,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,0.517,0.02924,0.04005,0.03772,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,0.584,0.03490,0.04825,0.04465,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


In [3]:
# Dataset schema and column types
parkinsons_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              195 non-null    str    
 1   MDVP:Fo(Hz)       195 non-null    float64
 2   MDVP:Fhi(Hz)      195 non-null    float64
 3   MDVP:Flo(Hz)      195 non-null    float64
 4   MDVP:Jitter(%)    195 non-null    float64
 5   MDVP:Jitter(Abs)  195 non-null    float64
 6   MDVP:RAP          195 non-null    float64
 7   MDVP:PPQ          195 non-null    float64
 8   Jitter:DDP        195 non-null    float64
 9   MDVP:Shimmer      195 non-null    float64
 10  MDVP:Shimmer(dB)  195 non-null    float64
 11  Shimmer:APQ3      195 non-null    float64
 12  Shimmer:APQ5      195 non-null    float64
 13  MDVP:APQ          195 non-null    float64
 14  Shimmer:DDA       195 non-null    float64
 15  NHR               195 non-null    float64
 16  HNR               195 non-null    float64
 17  status  

In [4]:
# Summary statistics for voice acoustic features
parkinsons_data.describe().T


,count,mean,std,min,25%,50%,75%,max
MDVP:Fo(Hz),195.0,154.228641,41.390065,88.333000,117.572000,148.790000,182.769000,260.105000
MDVP:Fhi(Hz),195.0,197.104918,91.491548,102.145000,134.862500,175.829000,224.205500,592.030000
MDVP:Flo(Hz),195.0,116.324631,43.521413,65.476000,84.291000,104.315000,140.018500,239.170000
MDVP:Jitter(%),195.0,0.006220,0.004848,0.001680,0.003460,0.004940,0.007365,0.033160
MDVP:Jitter(Abs),195.0,0.000044,0.000035,0.000007,0.000020,0.000030,0.000060,0.000260
MDVP:RAP,195.0,0.003306,0.002968,0.000680,0.001660,0.002500,0.003835,0.021440
MDVP:PPQ,195.0,0.003446,0.002759,0.000920,0.001860,0.002690,0.003955,0.019580
Jitter:DDP,195.0,0.009920,0.008903,0.002040,0.004985,0.007490,0.011505,0.064330
MDVP:Shimmer,195.0,0.029709,0.018857,0.009540,0.016505,0.022970,0.037885,0.119080
MDVP:Shimmer(dB),195.0,0.282251,0.194877,0.085000,0.148500,0.221000,0.350000,1.302000


### Target Distribution & Feature Comparisons
Analyze the class distribution (`status: 1` = Parkinson's, `status: 0` = Healthy) and inspect average feature values grouped by disease state.

In [5]:
# Distribution of target recordings
print("=== Class Distribution (Voice Recordings) ===")
status_counts = parkinsons_data['status'].value_counts()
print(f"Parkinson's Positive (1): {status_counts[1]} ({status_counts[1]/len(parkinsons_data)*100:.1f}%)")
print(f"Healthy Control      (0): {status_counts[0]} ({status_counts[0]/len(parkinsons_data)*100:.1f}%)")

# Mean feature values grouped by disease status
print("\n=== Mean Feature Values Grouped by Status ===")
display(parkinsons_data.groupby('status').mean(numeric_only=True))


=== Class Distribution (Voice Recordings) ===
Parkinson's Positive (1): 147 (75.4%)
Healthy Control      (0): 48 (24.6%)

=== Mean Feature Values Grouped by Status ===


,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,MDVP:APQ,Shimmer:DDA,NHR,HNR,RPDE,DFA,spread1,spread2,D2,PPE
status,,,,,,,,,,,,,,,,,,,,,,
0,181.937771,223.636750,145.207292,0.003866,0.000023,0.001925,0.002056,0.005776,0.017615,0.162958,0.009504,0.010509,0.013305,0.028511,0.011483,24.678750,0.442552,0.695716,-6.759264,0.160292,2.154491,0.123017
1,145.180762,188.441463,106.893558,0.006989,0.000051,0.003757,0.003900,0.011273,0.033658,0.321204,0.017676,0.020285,0.027600,0.053027,0.029211,20.974048,0.516816,0.725408,-5.333420,0.248133,2.456058,0.233828


## 3. Subject Extraction & Data Leakage Prevention

### Why Extract Subject IDs?
In biomedical voice analysis, each patient provides multiple voice recordings. In this dataset, the `name` column contains patient identifiers formatted as `phon_R01_S<Subject_ID>_<Recording_Index>` (e.g., `phon_R01_S01_1`).

* **The Problem (Data Leakage)**: If we apply standard random train/test splitting, multiple recordings from the same individual would be split across both training and test sets. The classifier would inadvertently memorize speaker-specific voice traits (pitch baseline, timbre) rather than genuine Parkinsonian acoustic markers.
* **The Solution (Subject-Aware Splitting)**: We extract the unique `subject` identifier and group data by subject. This ensures all recordings from any given patient appear strictly in the training set OR the test set—never both.

In [6]:
# Extract Subject ID from the 'name' column
parkinsons_data['subject'] = parkinsons_data['name'].apply(lambda x: x.split('_')[2])

# Aggregate subject-level summary
subject_summary = parkinsons_data.groupby('subject')['status'].agg(
    recordings='count',
    status='first'
)

print(f"Total voice recordings : {len(parkinsons_data)}")
print(f"Total unique subjects  : {parkinsons_data['subject'].nunique()}")
print(f"Recordings per subject : min={subject_summary['recordings'].min()}, max={subject_summary['recordings'].max()}, avg={subject_summary['recordings'].mean():.1f}")

print("\n=== Unique Subjects by Diagnostic Class ===")
subj_counts = subject_summary['status'].value_counts()
print(f"Parkinson's Patients (1): {subj_counts[1]} subjects")
print(f"Healthy Controls     (0): {subj_counts[0]} subjects")


Total voice recordings : 195
Total unique subjects  : 32
Recordings per subject : min=6, max=7, avg=6.1

=== Unique Subjects by Diagnostic Class ===
Parkinson's Patients (1): 24 subjects
Healthy Controls     (0): 8 subjects


## 4. Subject-Aware Stratified Train/Test Split
Partition the dataset into an 80% training set and a 20% test set by stratifying at the subject level. This guarantees zero subject overlap between training and evaluation.

In [7]:
# Separate features (X), target (Y), and group labels
X = parkinsons_data.drop(columns=['name', 'status', 'subject'])
Y = parkinsons_data['status']
groups = parkinsons_data['subject']

# Subject-level Stratified Train/Test Split (80% train, 20% test)
subject_df = parkinsons_data[['subject', 'status']].drop_duplicates()
train_subjects, test_subjects = train_test_split(
    subject_df['subject'],
    test_size=0.2,
    stratify=subject_df['status'],
    random_state=2
)

train_mask = parkinsons_data['subject'].isin(train_subjects)
test_mask = parkinsons_data['subject'].isin(test_subjects)

X_train, Y_train = X[train_mask], Y[train_mask]
X_test, Y_test = X[test_mask], Y[test_mask]

# Summary of split
print(f"Total recordings : {X.shape[0]} across {subject_df.shape[0]} unique subjects")
print(f"Training set     : {X_train.shape[0]} recordings across {len(train_subjects)} subjects (Status: {Y_train.value_counts().to_dict()})")
print(f"Test set         : {X_test.shape[0]} recordings across {len(test_subjects)} subjects (Status: {Y_test.value_counts().to_dict()})")


Total recordings : 195 across 32 unique subjects
Training set     : 152 recordings across 25 subjects (Status: {1: 116, 0: 36})
Test set         : 43 recordings across 7 subjects (Status: {1: 31, 0: 12})


## 5. Model Architecture & Pipeline Construction

### Why Use a Scikit-Learn Pipeline (`StandardScaler` + `Linear SVC`)?
Support Vector Classifiers construct maximum-margin hyperplanes based on geometric distances. Features in this dataset have vastly different magnitudes (e.g., `MDVP:Fo(Hz)` ~ 100-250 vs. `MDVP:Jitter(Abs)` ~ 0.00001). Without standardization, large-magnitude features dominate the optimization objective.

Using an integrated `Pipeline`:
1. **Prevents Preprocessing Leakage**: `StandardScaler` fits parameters (mean and standard deviation) solely on training data and transforms test data using those fitted parameters.
2. **Atomic Deployment**: The scaler and classifier are serialized together into a single artifact, ensuring identical transformations during real-time inference.

In [8]:
# Construct the Pipeline: StandardScaler -> Linear SVC
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', svm.SVC(kernel='linear'))
])

print("Pipeline Structure:")
print(pipeline)


Pipeline Structure:
Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', SVC(kernel='linear'))])


In [9]:
# Train the Pipeline strictly on the training partition
pipeline.fit(X_train, Y_train)
print("Pipeline successfully trained on training subjects.")


Pipeline successfully trained on training subjects.


## 6. Comprehensive Model Evaluation

We evaluate model performance through:
1. **Training Performance**: Sanity check for convergence and baseline accuracy.
2. **Held-Out Test Set (Unseen Subjects)**: Evaluates generalization on a dedicated 20% test cohort.
3. **5-Fold Stratified Group Cross-Validation**: Validates model stability across all 5 folds without subject leakage.

In [10]:
# 1. Training Set Accuracy
X_train_prediction = pipeline.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)
print(f'Accuracy score on training data: {training_data_accuracy * 100:.2f}%')


Accuracy score on training data: 92.76%


In [11]:
# 2. Test Set Evaluation on Unseen Subjects
X_test_prediction = pipeline.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

precision = precision_score(Y_test, X_test_prediction)
recall = recall_score(Y_test, X_test_prediction)
f1 = f1_score(Y_test, X_test_prediction)
y_scores = pipeline.decision_function(X_test)
roc_auc = roc_auc_score(Y_test, y_scores)

print("=" * 55)
print("       HELD-OUT TEST EVALUATION (UNSEEN SUBJECTS)")
print("=" * 55)
print(f"  Accuracy            : {test_data_accuracy * 100:.2f}%")
print(f"  Precision           : {precision:.4f}")
print(f"  Recall (Sensitivity): {recall:.4f}")
print(f"  F1-Score            : {f1:.4f}")
print(f"  ROC-AUC Score       : {roc_auc:.4f}")
print("=" * 55)

# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print("\n--- Confusion Matrix ---")
print(cm)
print(f"  True Negatives (Healthy correctly identified)   : {cm[0,0]}")
print(f"  False Positives (Healthy misclassified as PD)    : {cm[0,1]}")
print(f"  False Negatives (PD missed by classifier)       : {cm[1,0]}")
print(f"  True Positives (PD correctly identified)        : {cm[1,1]}")

# Classification Report
print("\n--- Detailed Classification Report ---")
print(classification_report(Y_test, X_test_prediction, target_names=['Healthy (0)', "Parkinson's (1)"]))


       HELD-OUT TEST EVALUATION (UNSEEN SUBJECTS)
  Accuracy            : 79.07%
  Precision           : 0.8235
  Recall (Sensitivity): 0.9032
  F1-Score            : 0.8615
  ROC-AUC Score       : 0.5833

--- Confusion Matrix ---
[[ 6  6]
 [ 3 28]]
  True Negatives (Healthy correctly identified)   : 6
  False Positives (Healthy misclassified as PD)    : 6
  False Negatives (PD missed by classifier)       : 3
  True Positives (PD correctly identified)        : 28

--- Detailed Classification Report ---
                 precision    recall  f1-score   support

    Healthy (0)       0.67      0.50      0.57        12
Parkinson's (1)       0.82      0.90      0.86        31

       accuracy                           0.79        43
      macro avg       0.75      0.70      0.72        43
   weighted avg       0.78      0.79      0.78        43



### 5-Fold Stratified Group Cross-Validation
To rigorously evaluate generalization without subject leakage across the entire dataset, we use **5-Fold `StratifiedGroupKFold`** grouped by `subject`. Each fold evaluates on completely unseen subjects while preserving class distribution.

In [12]:
# 5-Fold Stratified Group Cross-Validation on the full dataset using Pipeline
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2)

cv_acc = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='accuracy')
cv_precision = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='precision')
cv_recall = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='recall')
cv_f1 = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='f1')
cv_roc = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='roc_auc')

print('=== 5-Fold Stratified Group Cross-Validation Results ===')
print(f'Accuracy : {cv_acc.mean()*100:.2f}% (+/- {cv_acc.std()*100:.2f}%)')
print(f'Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})')
print(f'Recall   : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})')
print(f'F1-Score : {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
print(f'ROC-AUC  : {cv_roc.mean():.4f} (+/- {cv_roc.std():.4f})')


=== 5-Fold Stratified Group Cross-Validation Results ===
Accuracy : 77.97% (+/- 12.89%)
Precision: 0.8398 (+/- 0.1218)
Recall   : 0.8818 (+/- 0.0715)
F1-Score : 0.8561 (+/- 0.0848)
ROC-AUC  : 0.7570 (+/- 0.1791)


## 7. Building a Predictive System
Test the end-to-end inference capability of the fitted pipeline on a sample feature vector.

In [13]:
# Sample input data (22 biomedical voice features)
input_data = (197.07600, 206.89600, 192.05500, 0.00289, 0.00001, 0.00166, 0.00168, 0.00498, 0.01098, 0.09700, 0.00563, 0.00680, 0.00802, 0.01689, 0.00339, 26.77500, 0.422229, 0.741367, -7.348300, 0.177551, 1.743867, 0.085569)

# Convert to DataFrame with feature column names
input_df = pd.DataFrame([input_data], columns=X.columns)

prediction = pipeline.predict(input_df)
decision_score = pipeline.decision_function(input_df)[0]

print(f"Decision Function Score: {decision_score:.4f}")
print(f"Prediction Output: {prediction[0]}")

if prediction[0] == 0:
    print("Diagnostic Result: The person does not have Parkinson's disease (Healthy Control)")
else:
    print("Diagnostic Result: The person has Parkinson's disease")


Decision Function Score: -0.9616
Prediction Output: 0
Diagnostic Result: The person does not have Parkinson's disease (Healthy Control)


## 8. Saving the Trained Pipeline
Export the complete fitted pipeline (StandardScaler + Linear SVC) to disk for deployment in the Streamlit application.

In [14]:
# Saving the complete Pipeline (scaler + classifier)
filename = '../saved_models/parkinsons_pipeline.joblib'
joblib.dump(pipeline, filename)
print(f'Pipeline saved to {filename}')


Pipeline saved to ../saved_models/parkinsons_pipeline.joblib
